# Hangman — AI Edition

เกม Hangman 3 โหมด:
- **AI Assist** — ผู้เล่นเดาเอง แต่ AI แนะนำ top-5 ตัวอักษรก่อนทุก turn
- **AI Opponent** — AI เล่นเองทั้งหมด ผู้เล่นแค่ดู
- **VS AI** — ผู้เล่นกับ AI สลับกันเดาคำของอีกฝ่าย ใครผิดครบ 6 ก่อน = แพ้

> ต้องเทรนโมเดลก่อน (`train_models.py` -> `eval_models.py`) จึงจะใช้งานได้

In [ ]:
import random
import sys
import os

sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'codes'))
from predict import predict

# ============================================================
# ASCII Art
# ============================================================

HANGMAN_STAGES = [
    """
  +---+
  |   |
      |
      |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
      |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
  |   |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|   |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
      |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
 /    |
      |
=========""",
    """
  +---+
  |   |
  O   |
 /|\\  |
 / \\  |
      |
========="""
]

MAX_WRONG = len(HANGMAN_STAGES) - 1
W = 40

# ============================================================
# Word List
# ============================================================

WORD_CATEGORIES = {
    "animals":   ["elephant", "giraffe", "penguin", "dolphin", "cheetah",
                  "kangaroo", "crocodile", "butterfly", "octopus", "flamingo"],
    "fruits":    ["strawberry", "pineapple", "watermelon", "blueberry", "mango",
                  "avocado", "raspberry", "pomegranate", "apricot", "coconut"],
    "countries": ["thailand", "australia", "brazil", "canada", "germany",
                  "japan", "mexico", "norway", "portugal", "sweden"],
}

def choose_word():
    category = random.choice(list(WORD_CATEGORIES.keys()))
    word = random.choice(WORD_CATEGORIES[category])
    return word, category

# ============================================================
# Pretty Print Helpers
# ============================================================

def header(title):
    print("=" * W)
    print(title.center(W))
    print("=" * W)

def divider():
    print("-" * W)

def display_word(word, guessed):
    return "  ".join(c.upper() if c in guessed else "_" for c in word)

def print_state(label, word, guessed, wrong_count):
    divider()
    print(f"  {label}")
    divider()
    print(HANGMAN_STAGES[wrong_count])
    print()
    print(f"  Word    :  {display_word(word, guessed)}")
    wrong_letters = sorted(guessed - set(word))
    wrong_str = "  ".join(wrong_letters).upper() if wrong_letters else "-"
    lives = MAX_WRONG - wrong_count
    print(f"  Wrong   :  {wrong_str}")
    print(f"  Lives   :  {'[ ]' * lives}{'[X]' * wrong_count}  ({wrong_count}/{MAX_WRONG})")
    divider()

## Mode 1 — AI Assist
ผู้เล่นเดาเอง แต่ AI แนะนำ top-5 ตัวอักษรก่อนทุก turn

In [2]:
def play_ai_assist():
    header("HANGMAN  —  AI ASSIST MODE")

    word, category = choose_word()
    guessed = set()
    wrong_count = 0

    print(f"  Category : {category.upper()}")
    print(f"  Letters  : {len(word)}")
    divider()

    while wrong_count < MAX_WRONG:
        pattern = "".join(c if c in guessed else "_" for c in word)

        if "_" not in pattern:
            print_state("YOUR BOARD", word, guessed, wrong_count)
            print(f"  RESULT  :  YOU WIN!")
            print(f"  Word was:  {word.upper()}")
            divider()
            return

        print_state("YOUR BOARD", word, guessed, wrong_count)

        wrong_set  = guessed - set(word)
        suggestions = predict(pattern, guessed & set(word), wrong_set)
        print(f"  AI Hint  :  {' | '.join(s.upper() for s in suggestions)}")
        divider()

        while True:
            guess = input("  Your guess > ").strip().lower()
            if len(guess) != 1 or not guess.isalpha():
                print("  Enter a single letter (a-z).")
            elif guess in guessed:
                print(f"  '{guess.upper()}' already guessed. Try again.")
            else:
                break

        guessed.add(guess)

        if guess in word:
            print(f"  [ HIT ]  '{guess.upper()}' is in the word!")
        else:
            wrong_count += 1
            print(f"  [ MISS]  '{guess.upper()}' is NOT in the word.")

    print_state("YOUR BOARD", word, guessed, wrong_count)
    print(f"  RESULT  :  GAME OVER")
    print(f"  Word was:  {word.upper()}")
    divider()


while True:
    play_ai_assist()
    if input("\n  Play again? (yes/no) > ").strip().lower() != "yes":
        print("  Thanks for playing! Goodbye!")
        divider()
        break

       HANGMAN  —  AI ASSIST MODE       
  Category : FRUITS
  Letters  : 11
----------------------------------------
----------------------------------------
  YOUR BOARD
----------------------------------------

  +---+
  |   |
      |
      |
      |
      |

  Word    :  _  _  _  _  _  _  _  _  _  _  _
  Wrong   :  -
  Lives   :  [ ][ ][ ][ ][ ][ ]  (0/6)
----------------------------------------
  AI Hint  :  R | N | E | I | T
----------------------------------------
  Enter a single letter (a-z).
  Enter a single letter (a-z).
  Enter a single letter (a-z).
  [ HIT ]  'A' is in the word!
----------------------------------------
  YOUR BOARD
----------------------------------------

  +---+
  |   |
      |
      |
      |
      |

  Word    :  _  _  _  _  _  _  A  _  A  _  _
  Wrong   :  -
  Lives   :  [ ][ ][ ][ ][ ][ ]  (0/6)
----------------------------------------
  AI Hint  :  O | R | T | N | E
----------------------------------------
  [ HIT ]  'O' is in the word!
-----------

## Mode 2 — AI Opponent
AI เล่นเองทั้งหมด ผู้เล่นแค่ดู

In [ ]:
def play_ai_opponent(secret=None):
    header("HANGMAN  —  AI OPPONENT MODE")

    if secret:
        word, category = secret.lower(), "custom"
    else:
        word, category = choose_word()

    guessed = set()
    wrong_count = 0

    print(f"  Category : {category.upper()}")
    print(f"  Letters  : {len(word)}  ( {'_ ' * len(word)})")
    divider()
    input("  Press Enter to watch AI play...")

    while wrong_count < MAX_WRONG:
        pattern = "".join(c if c in guessed else "_" for c in word)

        if "_" not in pattern:
            print_state("AI BOARD", word, guessed, wrong_count)
            print(f"  RESULT  :  AI WINS in {len(guessed)} guesses!")
            print(f"  Word was:  {word.upper()}")
            divider()
            return

        print_state("AI BOARD", word, guessed, wrong_count)

        wrong_set   = guessed - set(word)
        suggestions = predict(pattern, guessed & set(word), wrong_set)
        guess       = suggestions[0]

        print(f"  AI Rank  :  {' | '.join(s.upper() for s in suggestions)}")
        print(f"  AI Guess :  {guess.upper()}")
        divider()
        input("  Press Enter to continue...")

        guessed.add(guess)

        if guess in word:
            print(f"  [ HIT ]  '{guess.upper()}' is in the word!")
        else:
            wrong_count += 1
            print(f"  [ MISS]  '{guess.upper()}' is NOT in the word.")

    print_state("AI BOARD", word, guessed, wrong_count)
    print(f"  RESULT  :  AI LOSES")
    print(f"  Word was:  {word.upper()}")
    divider()


SECRET_WORD = None   # เช่น "elephant"

while True:
    play_ai_opponent(SECRET_WORD)
    if input("\n  Watch again? (yes/no) > ").strip().lower() != "yes":
        print("  Thanks for watching! Goodbye!")
        divider()
        break

## Mode 3 — VS AI
ผู้เล่นกับ AI สลับกันเดาคำของอีกฝ่าย ใครผิดครบ 6 ก่อน = แพ้

**กติกา:**
- ผู้เล่นตั้งคำให้ AI เดา และ AI สุ่มคำให้ผู้เล่นเดา
- สลับ turn ไปเรื่อยๆ จนฝ่ายใดฝ่ายหนึ่งผิดครบ 6 ครั้ง

In [3]:
def vs_ai_turn_player(word, guessed, wrong_count):
    pattern = "".join(c if c in guessed else "_" for c in word)
    if "_" not in pattern or wrong_count >= MAX_WRONG:
        return guessed, wrong_count, True

    print_state("YOUR TURN  —  Guess AI's word", word, guessed, wrong_count)

    while True:
        guess = input("  Your guess > ").strip().lower()
        if len(guess) != 1 or not guess.isalpha():
            print("  Enter a single letter (a-z).")
        elif guess in guessed:
            print(f"  '{guess.upper()}' already guessed. Try again.")
        else:
            break

    guessed.add(guess)
    if guess in word:
        print(f"  [ HIT ]  '{guess.upper()}' is in the word!")
    else:
        wrong_count += 1
        print(f"  [ MISS]  '{guess.upper()}' is NOT in the word.")

    done = "_" not in "".join(c if c in guessed else "_" for c in word) or wrong_count >= MAX_WRONG
    return guessed, wrong_count, done


def vs_ai_turn_ai(word, guessed, wrong_count):
    pattern = "".join(c if c in guessed else "_" for c in word)
    if "_" not in pattern or wrong_count >= MAX_WRONG:
        return guessed, wrong_count, True

    print_state("AI TURN  —  AI guesses your word", word, guessed, wrong_count)

    wrong_set   = guessed - set(word)
    suggestions = predict(pattern, guessed & set(word), wrong_set)
    guess       = suggestions[0]

    print(f"  AI Rank  :  {' | '.join(s.upper() for s in suggestions)}")
    print(f"  AI Guess :  {guess.upper()}")
    divider()
    input("  Press Enter to continue...")

    guessed.add(guess)
    if guess in word:
        print(f"  [ HIT ]  '{guess.upper()}' is in the word!")
    else:
        wrong_count += 1
        print(f"  [ MISS]  '{guess.upper()}' is NOT in the word.")

    done = "_" not in "".join(c if c in guessed else "_" for c in word) or wrong_count >= MAX_WRONG
    return guessed, wrong_count, done


def print_result(winner, ai_word, player_word, p_wrong, a_wrong):
    divider()
    print("  GAME OVER".center(W))
    divider()
    print(f"  AI's word    :  {ai_word.upper()}")
    print(f"  Your word    :  {player_word.upper()}")
    print(f"  Your wrong   :  {p_wrong}/{MAX_WRONG}")
    print(f"  AI wrong     :  {a_wrong}/{MAX_WRONG}")
    divider()
    print(f"  WINNER  :  {winner}".center(W))
    divider()


def play_vs_ai():
    header("HANGMAN  —  VS AI MODE")

    player_word = input("  Set a word for AI to guess > ").strip().lower()
    while not player_word.isalpha() or len(player_word) < 3:
        player_word = input("  Invalid. Enter a word (letters only, min 3) > ").strip().lower()

    ai_word, category = choose_word()

    print(f"  AI's word category : {category.upper()} ({len(ai_word)} letters)")
    print(f"  Your word          : {'_' * len(player_word)} ({len(player_word)} letters)")
    divider()

    p_guessed, p_wrong = set(), 0
    a_guessed, a_wrong = set(), 0

    while True:
        p_guessed, p_wrong, p_done = vs_ai_turn_player(ai_word, p_guessed, p_wrong)

        if p_wrong >= MAX_WRONG:
            print_result("AI", ai_word, player_word, p_wrong, a_wrong)
            return
        if p_done:
            print_result("YOU", ai_word, player_word, p_wrong, a_wrong)
            return

        a_guessed, a_wrong, a_done = vs_ai_turn_ai(player_word, a_guessed, a_wrong)

        if a_wrong >= MAX_WRONG:
            print_result("YOU", ai_word, player_word, p_wrong, a_wrong)
            return
        if a_done:
            print_result("AI", ai_word, player_word, p_wrong, a_wrong)
            return


while True:
    play_vs_ai()
    if input("\n  Play again? (yes/no) > ").strip().lower() != "yes":
        print("  Thanks for playing! Goodbye!")
        divider()
        break

         HANGMAN  —  VS AI MODE         


  AI's word category : COUNTRIES (8 letters)
  Your word          : ______ (6 letters)
----------------------------------------
----------------------------------------
  YOUR TURN  —  Guess AI's word
----------------------------------------

  +---+
  |   |
      |
      |
      |
      |

  Word    :  _  _  _  _  _  _  _  _
  Wrong   :  -
  Lives   :  [ ][ ][ ][ ][ ][ ]  (0/6)
----------------------------------------
  [ HIT ]  'T' is in the word!
----------------------------------------
  AI TURN  —  AI guesses your word
----------------------------------------

  +---+
  |   |
      |
      |
      |
      |

  Word    :  _  _  _  _  _  _
  Wrong   :  -
  Lives   :  [ ][ ][ ][ ][ ][ ]  (0/6)
----------------------------------------
  AI Rank  :  A | N | O | R | E
  AI Guess :  A
----------------------------------------
  [ HIT ]  'A' is in the word!
----------------------------------------
  YOUR TURN  —  Guess AI's word
----------------------------------------

  +---+
  |   |
   